# درس ۰۳ - الگوهای طراحی عامِل‌محور

در این درس، سه الگوی طراحی پایه‌ای برای ساخت عامل‌های هوش مصنوعی مؤثر را بررسی می‌کنیم:

۱. **دستورات واضح برای عامل** — ایجاد دستورالعمل‌های دقیق و تعیین‌کننده نقش که رفتار عامل را هدایت می‌کنند  
۲. **خروجی ساختاریافته با مدل‌های Pydantic** — اطمینان از اینکه عامل‌ها داده‌های قابل پیش‌بینی و تأییدشده بازمی‌گردانند  
۳. **عامل‌های با مسئولیت تک** — طراحی عامل‌های متمرکز که هر کدام یک کار را به‌خوبی انجام می‌دهند

هر الگو را در سناریوی **پیشنهاد مقصد سفر** به‌کار می‌بریم و به‌تدریج سیستمی می‌سازیم که بتواند مقصدها را پیشنهاد دهد، موجودی را بررسی کند و امور لجستیکی را مدیریت کند.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## الگوی ۱: دستورالعمل‌های روشن برای نماینده

تاثیرگذارترین الگو همچنین ساده‌ترین است: نوشتن دستورالعمل‌های واضح و دقیق برای نماینده‌تان.

دستورالعمل‌های خوب تعریف می‌کنند:
- **چه کسی** نماینده است (شخصیت و لحن)
- **چه کاری** باید انجام دهد (مسئولیت‌های مرحله به مرحله)
- **چگونه** باید رفتار کند (محدودیت‌ها و سبک)

در ادامه، یک نماینده هماهنگ‌کننده سفر با دستورالعمل‌های صریح ایجاد می‌کنیم که هر پاسخی را که تولید می‌کند شکل می‌دهد.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## الگو ۲: خروجی ساختاریافته با مدل‌های Pydantic

متن آزاد برای گفتگو مفید است، اما سیستم‌های پایین‌دستی به داده‌های ساختاریافته نیاز دارند.
با ترکیب **مدل‌های Pydantic** با یک **تابع ابزار**، ما می‌توانیم:

- یک ساختار دقیق برای خروجی عامل تعریف کنیم
- پاسخ‌ها را به‌طور خودکار اعتبارسنجی کنیم
- نتایج عامل را به طور قابل اعتماد در منطق برنامه ادغام کنیم

همچنین ابزاری معرفی می‌کنیم که جزئیات مقصد را بازمی‌گرداند تا عامل توصیه‌های خود را بر داده‌های واقعی پایه‌گذاری کند.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## الگو ۳: عامل‌های با مسئولیت واحد

کارهای پیچیده از تقسیم کار بین چند عامل متمرکز که هر کدام مسئولیت واحدی دارند، بهره می‌برند:

- یک **کارشناس مقصد** که درباره مکان‌ها و در دسترس بودن اطلاع دارد
- یک **برنامه‌ریز لجستیک** که پروازها، هتل‌ها و برنامه سفرها را مدیریت می‌کند

این بازتاب اصل مهندسی نرم‌افزار *تفکیک مسئولیت‌ها* است — هر عامل به صورت مستقل آسان‌تر آزمایش، نگهداری و بهبود می‌یابد.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصه

در این درس سه الگوی طراحی عامل‌محور را در یک سناریوی توصیه‌گر سفر به کار بردیم:

| الگو | ایده کلیدی | مزیت |
|---|---|---|
| **دستورالعمل‌های واضح** | تعریف شخصیت، مسئولیت‌ها و محدودیت‌ها از پیش | رفتار عامل سازگار و هماهنگ با برند |
| **خروجی ساختاریافته** | استفاده از مدل‌های Pydantic به عنوان فرمت پاسخ | نتایج معتبر و قابل خواندن توسط ماشین |
| **مسئولیت‌پذیری منفرد** | دادن یک وظیفه متمرکز به هر عامل | آسان‌تر برای تست، نگهداری و ترکیب |

این الگوها به صورت طبیعی ترکیب می‌شوند — شما می‌توانید دستورالعمل‌های واضح را با خروجی ساختاریافته داخل یک عامل با مسئولیت منفرد ترکیب کنید تا سیستم‌هایی قوی و آماده تولید بسازید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه ماشینی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نواقصی باشند. سند اصلی به زبان بومی خود، منبع معتبر و اصلی محسوب می‌شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سوءتفاهم یا تفسیر نادرستی ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
